# core

## my_llm.py

In [1]:
from core.my_llm import MyLLM
my_llm = MyLLM(
    provider="modelscope"
)

input = "who are you?"
messages = [
    {"role": "user", "content": input}
]

print(my_llm.generate(
    messages=messages
))

----------llm invoking start----------
messages:
[{'role': 'user', 'content': 'who are you?'}]
output:
Hello! I'm Qwen, a large-scale language model independently developed by the Tongyi Lab under Alibaba Group. I can assist you with answering questions, writing, logical reasoning, programming, and more. I support 100 languages, including but not limited to Chinese, English, German, French, Spanish, etc., meeting international usage needs. If you have any questions or need help, feel free to let me know anytime!
----------llm invoking over----------

Hello! I'm Qwen, a large-scale language model independently developed by the Tongyi Lab under Alibaba Group. I can assist you with answering questions, writing, logical reasoning, programming, and more. I support 100 languages, including but not limited to Chinese, English, German, French, Spanish, etc., meeting international usage needs. If you have any questions or need help, feel free to let me know anytime!


In [2]:
from core.my_llm import MyLLM
my_llm = MyLLM(
    provider="qwen"
)

input = "who are you?"
messages = [
    {"role": "user", "content": input}
]

my_llm.generate(
    messages=messages
)

----------llm invoking start----------
messages:
[{'role': 'user', 'content': 'who are you?'}]
output:
I'm Qwen, a large-scale language model developed by Tongyi Lab. I can help you answer questions, write stories, create documents, compose emails, script videos, logical reasoning, programming, and more. I can also express opinions and play games. My training data comes entirely from the historical accumulation within Tongyi Lab, and I don't have real-time access to the internet or personal data unless explicitly provided in our conversation.

Feel free to ask me anything—I'm here to help! 😊
----------llm invoking over----------



"I'm Qwen, a large-scale language model developed by Tongyi Lab. I can help you answer questions, write stories, create documents, compose emails, script videos, logical reasoning, programming, and more. I can also express opinions and play games. My training data comes entirely from the historical accumulation within Tongyi Lab, and I don't have real-time access to the internet or personal data unless explicitly provided in our conversation.\n\nFeel free to ask me anything—I'm here to help! 😊"

## agent.py

In [3]:
from core.agent import Agent
from core.my_llm import MyLLM
from core.message import Message
from core.config import Config

class TestAgent(Agent):
    def run(self, input:str, **kwargs) -> str:
        # 更新输入
        user_message = Message(
            role="user",
            content=input
        )
        self.update_history(message=user_message)
        

        # llm生成响应
        respone = self.llm.generate(
            messages=self._history
        )

        # 更新输出
        assistant_message = Message(
            role="assistant",
            content=respone
        )


In [4]:
my_llm = MyLLM()
test_agent = TestAgent(
    llm = my_llm
)

input = "who are you"
test_agent.run(
    input=input
)

----------llm invoking start----------
messages:
[Message(role='user', content='who are you', timestamp=datetime.datetime(2026, 2, 19, 11, 29, 48, 91134), metadata={})]
output:
Hi there! 😊 I'm Qwen, a large-scale language model developed by Tongyi Lab. Think of me as your curious, helpful, and always-learning AI companion—I can chat with you, answer questions (whether they're about science, history, daily life, or even quirky "what if" scenarios), help write stories or emails, explain complex ideas in simple terms, assist with coding, and much more.

I don’t have personal experiences or emotions—but I’m designed to be respectful, thoughtful, and genuinely interested in helping *you* learn, create, or solve problems. Whether you're studying, working, brainstorming, or just feeling curious, I'm here to support you.

What would you like to explore or do together? 🌟
----------llm invoking over----------



# agents

## simple_agent.py

In [5]:
from agents.simple_agent import SimpleAgent
from core.my_llm import MyLLM


my_simple_agent = SimpleAgent(
    llm = MyLLM(),
    name="test agent"
)

my_simple_agent.run(
    input="计算 1 + 2 + 3 + 4*7"
)

====================test agent processing start: 计算 1 + 2 + 3 + 4*7====================
----------llm invoking start----------
messages:
[{'role': 'system', 'content': 'You are a helpful AI assistant.\n\n## Output contract (STRICT)\nOutput ONLY a JSON object. No extra text, no explanation before or after.\nFormat: {"type": "final", "final": "your answer"}'}, {'role': 'user', 'content': '计算 1 + 2 + 3 + 4*7'}]
output:
{"type": "final", "final": "36"}
----------llm invoking over----------

====================test agent processing end: 计算 1 + 2 + 3 + 4*7====================


'36'

In [6]:
from agents.simple_agent import SimpleAgent
from core.my_llm import MyLLM
from tools.builtin.search import search, SEARCH_DESCRIPTION
from tools.registry import ToolRegistry

# 增加工具
tool_registry =  ToolRegistry()
tool_registry.registerTool(
    name="search",
    description=SEARCH_DESCRIPTION,
    function=search
)

my_simple_agent = SimpleAgent(
    llm = MyLLM(),
    name="test agent",
    enable_tool_calling=True,
    tool_registry=tool_registry
)

my_simple_agent.run(
    input="what's the price of gold"
)

====================test agent processing start: what's the price of gold====================
----------llm invoking start----------
messages:
[{'role': 'system', 'content': 'You are a helpful AI assistant.\n\n## Available tools\n- finish(conclusion: str): Call this action when you think it\'s time to end, args is your conclusion\n- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.\n\n## Output contract (STRICT)\nYou MUST output exactly one JSON object and nothing else (no markdown, no extra text).\n\nSchema:\n{\n  "type": "tool_call" | "final",\n  "tool_calls": [{"name": string, "args": object}],\n  "final": string\n}\n\nRules:\n1) If you need a tool, set type="tool_call" and provide tool_calls.\n2) Call tools ONLY from the available tool list. Never invent tool names.\n3) Do NOT hallucinate tool outputs. Use tools to get results.\n4) After a tool_call, you will

'The current price of gold is approximately $4,990.60 per ounce.'

## react_agent.py

In [1]:
from core import Agent, Message, Config, MyLLM
from agents.react_agent import ReActAgent
from tools.registry import ToolRegistry
from tools.builtin.search import SEARCH_DESCRIPTION, search

# 整理工具
tool_registry = ToolRegistry()
tool_registry.registerTool(
    name='search',
    description=SEARCH_DESCRIPTION,
    function=search
)

my_react_agent = ReActAgent(
    llm = MyLLM(),
    tool_registry=tool_registry,
)

my_react_agent.run(
    input="First query the price of gold, then query the exchange rate between RMB and US dollar, then convert to get the gold price in RMB."
)

====================anonymous processing start: First query the price of gold, then query the exchange rate between RMB and US dollar, then convert to get the gold price in RMB.====================
----------step: 0----------
----------llm invoking start----------
messages:
[{'role': 'user', 'content': 'You are an AI assistant that can use tools.\n\n## Available Tools\n- finish(conclusion: str): Call this action when you think it\'s time to end, args is your conclusion\n- search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information.\n\n## Output Contract (STRICT)\nYou MUST output exactly ONE JSON object and NOTHING else (no markdown, no extra text).\n\nSchema:\n{\n  "type": "tool_call" | "final",\n  "tool": {\n    "name": string,\n    "args": object\n  } | null,\n  "final": string | null,\n  "notes": string | null\n}\n\nRules:\n1) Execute ONLY ONE step at a time.\n2) If you 

'The current gold price is approximately $4,990.60 per ounce. With an exchange rate of 1 USD = 6.9089 CNY, the gold price in RMB is approximately ¥34,482.50 per ounce.'